# ARIMA Forecast – LastFM Sessions

**Exercise 3 – ML Challenge**

- Data processing: **PySpark**
- Forecasting: **ARIMA(1,1,1)** via statsmodels (pandas)

Steps:
1. Load data with PySpark and build sessions (gap > 20 min = new session)
2. Top 10 songs in the top 50 longest sessions by track count
3. Select top 1 user by number of sessions
4. Aggregate weekly sessions → convert to pandas → ARIMA(1,1,1) forecast for next 3 months

## 1. Start Spark session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName('LastFM_ARIMA') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')

## 2. Load data

In [ ]:
DATA_PATH = '../data/userid-timestamp-artid-artname-traid-traname.tsv'

df = spark.read.csv(
    DATA_PATH,
    sep='\t',
    header=False,
    inferSchema=False
).toDF('userid', 'timestamp', 'artid', 'artname', 'traid', 'traname')

df = df.withColumn('timestamp', F.to_timestamp('timestamp'))

print(f'Total rows: {df.count():,}')
df.show(5, truncate=False)

## 3. Build sessions (gap > 20 min = new session)

In [ ]:
GAP_SECONDS = 20 * 60

w_user = Window.partitionBy('userid').orderBy('timestamp')

df = (
    df
    .withColumn('prev_ts', F.lag('timestamp').over(w_user))
    .withColumn(
        'new_session',
        F.when(
            F.col('prev_ts').isNull() |
            (F.unix_timestamp('timestamp') - F.unix_timestamp('prev_ts') > GAP_SECONDS),
            1
        ).otherwise(0)
    )
    .withColumn('session_id', F.sum('new_session').over(w_user))
    .withColumn('session_key', F.concat_ws('_', 'userid', 'session_id'))
)

total_sessions = df.select('session_key').distinct().count()
print(f'Total sessions: {total_sessions:,}')

## 4. Top 10 songs in the top 50 longest sessions by track count

In [ ]:
session_sizes = (
    df.groupBy('session_key')
    .count()
    .withColumnRenamed('count', 'track_count')
)

top50_keys = [
    r['session_key']
    for r in session_sizes.orderBy(F.desc('track_count')).limit(50).collect()
]

top10_songs = (
    df.filter(F.col('session_key').isin(top50_keys))
    .groupBy('traname')
    .count()
    .withColumnRenamed('count', 'plays')
    .orderBy(F.desc('plays'))
    .limit(10)
)

print('Top 10 songs in top 50 longest sessions:')
top10_songs.show(truncate=False)

## 5. Top 1 user by number of sessions

In [ ]:
user_sessions = (
    df.select('userid', 'session_key').distinct()
    .groupBy('userid')
    .count()
    .withColumnRenamed('count', 'n_sessions')
    .orderBy(F.desc('n_sessions'))
)

top_user_row = user_sessions.first()
top_user   = top_user_row['userid']
top_n_sess = top_user_row['n_sessions']
print(f'Top user: {top_user}  -  {top_n_sess:,} sessions')

## 6. Weekly session count for the top user

In [ ]:
weekly_spark = (
    df.filter(F.col('userid') == top_user)
    .groupBy('session_key')
    .agg(F.min('timestamp').alias('session_start'))
    .withColumn('week', F.date_trunc('week', 'session_start'))
    .groupBy('week')
    .agg(F.count('session_key').alias('n_sessions'))
    .orderBy('week')
)

weekly_spark.show(10)

## 7. Convert to pandas and plot

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

weekly_pd = weekly_spark.toPandas()
weekly_pd['week'] = pd.to_datetime(weekly_pd['week']).dt.tz_localize(None)
weekly = weekly_pd.set_index('week')['n_sessions'].asfreq('W')

print(f'Date range: {weekly.index.min().date()} to {weekly.index.max().date()}')

weekly.plot(title=f'Weekly sessions - user {top_user}', figsize=(12, 4))
plt.ylabel('Sessions')
plt.tight_layout()
plt.show()

## 8. ARIMA(1,1,1) forecast - next 3 months (~13 weeks)

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')

model  = ARIMA(weekly.fillna(0), order=(1, 1, 1))
result = model.fit()
print(result.summary())

In [ ]:
FORECAST_WEEKS = 13

forecast = result.get_forecast(steps=FORECAST_WEEKS)
fc_mean  = forecast.predicted_mean
fc_ci    = forecast.conf_int()

fig, ax = plt.subplots(figsize=(14, 5))
weekly.plot(ax=ax, label='Historical', color='steelblue')
fc_mean.plot(ax=ax, label='Forecast', color='darkorange', linestyle='--')
ax.fill_between(
    fc_ci.index,
    fc_ci.iloc[:, 0],
    fc_ci.iloc[:, 1],
    color='darkorange', alpha=0.2, label='95% CI'
)
ax.set_title(f'ARIMA(1,1,1) - Weekly sessions forecast (next 3 months) - user {top_user}')
ax.set_ylabel('Sessions')
ax.legend()
plt.tight_layout()
plt.show()

print('\nForecast values:')
fc_mean.rename('predicted_sessions').to_frame().join(fc_ci).round(2)

In [ ]:
spark.stop()